# Session 9 · Exercises

*Course material, licence CC BY 4.0.*

Work in pairs. The lecturer chooses which exercises to set; there are more than fit into 30 minutes. *Paper* and *Predict* tasks are answered in the markdown cell below the task, before any code is run; *Code* tasks in the empty code cells. Exercises marked *Stretch* are for pairs who finish early.

## Setup

Run these cells once before the exercises that use data.

In [ ]:
# Run the code from the repository root, where the case-study paths start.
import os
from pathlib import Path

os.chdir(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").is_dir()))

In [ ]:
import json

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold

listings = pd.read_parquet("case-study/data/airbnb/listings.parquet")
reviews = pd.read_parquet("case-study/data/airbnb/reviews_monthly.parquet")   # one row per listing and month

# Short-stay listings with a price, as in the lecture notebook
bnb = listings[(listings["minimum_nights"] < 28) & listings["price"].between(10, 1000)].reset_index(drop=True)
y, hosts = np.log(bnb["price"]), bnb["host_id"]
cv = GroupKFold(5)                                          # whole hosts per fold
print(len(bnb), "priced listings;", len(reviews), "listing-months of reviews")

In [ ]:
# The demand question of the lecture (section 6): features as of 1 July 2025, target = reviews in the next 12 months
cutoff = pd.Timestamp("2025-07-01")
past = reviews[reviews["month"] < cutoff]
future = reviews[(reviews["month"] >= cutoff) & (reviews["month"] < "2026-07-01")]
g = past.groupby("listing_id")
demand = pd.DataFrame({
    "rev_total": g["n_reviews"].sum(),
    "rev_last12": past[past["month"] >= cutoff - pd.DateOffset(months=12)].groupby("listing_id")["n_reviews"].sum(),
}).fillna({"rev_last12": 0})
demand = listings.set_index("id")[["host_id", "accommodates", "room_type"]].join(demand, how="inner")
demand["target"] = future.groupby("listing_id")["n_reviews"].sum().reindex(demand.index).fillna(0)
y_d = np.log1p(demand["target"])
print(len(demand), "listings with a review before the cut-off")

## Part 1 · Feature construction without leakage

### 1.1 · Is December far from January?

*Paper, 5 min*

> As numbers 1–12, December and January are 11 apart. How would you encode the month so that a linear model sees them as neighbours? Compute the encoding for January, July and December.

*Source:* [scikit-learn example: Time-related feature engineering](https://scikit-learn.org/stable/auto_examples/applications/plot_cyclical_feature_engineering.html)

Your answer:

### 1.2 · Is a degree a degree?

*Paper, 5 min*

> Is one degree of latitude as long as one degree of longitude in Berlin (latitude 52.5°)? What does this mean for a distance computed from raw coordinates?

*Source:* [Longitude, Wikipedia (length of a degree)](https://en.wikipedia.org/wiki/Longitude)

Your answer:

### 1.3 · Target encoding of a category seen once

*Paper, 5 min*

> A neighbourhood appears in exactly one training listing. What is its naive target encoding, and what does the model learn from it? How do out-of-fold encoding and smoothing help?

*Source:* [scikit-learn: TargetEncoder and cross fitting](https://scikit-learn.org/stable/modules/preprocessing.html#target-encoder)

Your answer:

### 1.4 · Guests × room type

*Class, 5 min*

> Can a linear model capture "an extra guest adds more to an entire home than to a room" without an interaction term?

*Source:* original exercise for this course; background: [Interaction (statistics), Wikipedia](https://en.wikipedia.org/wiki/Interaction_(statistics))

Your answer:

### 1.5 · Spot the leak

*Paper, 10 min*

> Ten columns for "will this listing be busy next year?", predicted on 1 July 2025 from the snapshot of 26 June 2026. Which are not known at the time of prediction?
>
> `accommodates`, `room_type`, `neighbourhood`, `host_since`, `amenities`, `estimated_revenue_l365d`, `number_of_reviews_ltm`, `last_review`, `availability_365`, `first_review`

*Source:* [Kaufman et al. (2012), "Leakage in data mining", *ACM TKDD*](https://doi.org/10.1145/2382577.2382579)

Your answer:

### 1.6 · The husky and the snow

*Class, 10 min*

> Ribeiro et al. (2016) deliberately trained an image classifier that told wolves from huskies by the snow in the background. Which "snow" could hide in our data: listing ID, scrape date, the host's other listings?

*Source:* [Ribeiro, Singh & Guestrin (2016), "Why should I trust you?"](https://arxiv.org/abs/1602.04938)

Your answer:

### 1.7 · Is demand growing or shrinking?

*Code, 25 min*

> The setup builds the demand table of the lecture with two past-only features (`rev_total`, `rev_last12`). Add a **trend** feature: the reviews in the last 6 months before the cut-off minus the reviews in the 6 months before those. Use only rows of `past`.
>
> Compare a `HistGradientBoostingRegressor(random_state=0)` with and without the trend feature (features: `rev_total`, `rev_last12`, `accommodates`, plus the trend) by host-grouped cross-validation (`cv`, `groups=demand["host_id"]`, target `y_d`). Does the trend help? Check one listing by hand.

*Source:* original exercise for this course; background: [Feast: point-in-time joins](https://docs.feast.dev/getting-started/concepts/point-in-time-joins)

### 1.8 · A count encoding that cannot leak

*Code, 20 min, *Stretch**

> Write a small transformer class (`BaseEstimator`, `TransformerMixin`) that learns in `fit` how often each `property_type` occurs in the training rows and in `transform` replaces each value by that count (0 for a type not seen in training).
>
> Put it in a `ColumnTransformer` next to `accommodates` and `bathrooms` (passed through), followed by a `HistGradientBoostingRegressor(random_state=0)`, and compare, by host-grouped cross-validation of the log price (`y`, `cv`, `groups=hosts`), with `OneHotEncoder(min_frequency=20, handle_unknown="infrequent_if_exist", sparse_output=False)` for `property_type`. Why must the counts be learned in `fit` and not computed on all rows?

*Source:* original exercise for this course; background: [scikit-learn: Developing scikit-learn estimators](https://scikit-learn.org/stable/developers/develop.html)